In [ ]:
%%bash
# v3: the v2 hybrid trained on asl-signs train + ASL Citizen train (mapped signs) with webcam
# augmentation, three seeds, selected and calibrated on validation only, exported as fp16 ONNX.
# No test split is read here: the asl-signs and ASL Citizen test sets and test_clips_v2 are
# evaluated once, locally (kaggle/ASL_CITIZEN.md).
#
# Settings: Accelerator GPU (T4), Internet On, Save Version > Save & Run All (Commit).
# Inputs: Competitions > Google - Isolated Sign Language Recognition; the ASL Citizen landmarks,
# either Your Work > Datasets > asl-citizen-landmarks-private or Your Work > Notebooks > the
# extraction notebook's merge run; and, unless this branch is pushed to GitHub, your private
# dataset edge-vision-src holding edge-vision-src.tar.gz (git archive of the commit to train).
# Resumable: finished seeds are skipped when an earlier version's output is added as an input.
# Python: Kaggle's own (3.13 at the time of writing) with its CUDA build of torch; every other
# package the code imports has a cp313 wheel at the pinned versions of requirements.txt, which
# are installed below. Licence: everything stays private (docs/DATA_LICENSES.md). Generated by
# scripts/build_kaggle_notebooks.py from kaggle/train_v3.sh.
set -u
WORK=/kaggle/working
INPUT=/kaggle/input
LOG="$WORK/train_v3.log"
REPO="$WORK/edge-vision"
BRANCH="main"            # used only when no edge-vision-src input is attached
MAX_HOURS=9              # no new seed starts after this, so export and packing fit in 12 h
START=$(date +%s)
mkdir -p "$WORK"
# find under the inputs without walking the 94k asl-signs parquet files or landmark shards
find_input() { find "$INPUT" \( -name train_landmarks -o -name shards \) -prune -o "$@" -print 2>/dev/null; }
versions() {
  python - <<'PYEOF'
import importlib, platform
print(f"python {platform.python_version()}")
for name in ("torch", "numpy", "pandas", "pyarrow", "pydantic", "yaml", "psutil", "onnx", "onnxruntime"):
    try:
        module = importlib.import_module(name)
        extra = ""
        if name == "torch":
            extra = f" (cuda {module.version.cuda}, available {module.cuda.is_available()})"
        print(f"{name} {getattr(module, '__version__', '?')}{extra}")
    except Exception as error:
        print(f"{name} MISSING: {type(error).__name__}: {error}")
PYEOF
}
{
  echo "=== $(date -u '+%F %T') UTC: start"
  if [ ! -f "$REPO/scripts/v3_kaggle.py" ]; then
    rm -rf "$REPO"
    ARCHIVE=$(find_input -name 'edge-vision-src*.tar.gz' | head -1)
    # Kaggle may unpack an uploaded archive, so also accept the unpacked tree of that dataset.
    TREE=$(find_input -path '*edge-vision-src*' -name pyproject.toml | head -1)
    if [ -n "$ARCHIVE" ]; then
      echo "source: $ARCHIVE"
      mkdir -p "$REPO" && tar -xzf "$ARCHIVE" -C "$REPO"
    elif [ -n "$TREE" ]; then
      echo "source: $(dirname "$TREE") (unpacked archive)"
      cp -r "$(dirname "$TREE")" "$REPO"
    else
      echo "source: git clone -b $BRANCH"
      git clone --depth 1 -b "$BRANCH" https://github.com/Swapnil007-Curious/edge-vision.git "$REPO"
    fi
  fi
  if [ ! -f "$REPO/scripts/v3_kaggle.py" ]; then
    echo "ERROR: the source has no scripts/v3_kaggle.py. Attach the edge-vision-src dataset (git"
    echo "archive of the v3 commit) or set BRANCH to a pushed branch that has the v3 code; stopping."
    exit 0
  fi
  cd "$REPO" || exit 0
  echo "--- versions in Kaggle's environment"
  versions
  # Kaggle's CUDA torch stays; the other packages are pinned to the versions tested locally.
  python -m pip install -q --disable-pip-version-check -r requirements.txt "onnx==1.23.2" \
    || echo "WARNING: pip could not install the pinned requirements; continuing with Kaggle's"
  python -c "import onnxruntime" 2>/dev/null || python -m pip install -q "onnxruntime==1.31.1"
  python -m pip install -q --disable-pip-version-check --no-deps -e . \
    || echo "ERROR: pip install of the repository failed"
  echo "--- versions used for training"
  versions
  python -c "import torch; print('GPU', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)); print('kernel check', (torch.ones(4, device='cuda') * 2).sum().item())" \
    || echo "WARNING: no usable GPU; training on the CPU is far too slow"

  # The inputs: asl-signs, the ASL Citizen landmarks (dataset or notebook output; smoke-test and
  # unfinished outputs are skipped) and finished runs of an earlier version of this notebook.
  echo "--- inputs under $INPUT"
  DATA_ROOT=""; LANDMARKS=""; PREVIOUS_RUNS=()
  eval "$(python scripts/v3_kaggle.py locate --input "$INPUT" --unpack-dir /tmp)"
  echo "asl-signs: ${DATA_ROOT:-NOT FOUND}"
  echo "ASL Citizen landmarks: ${LANDMARKS:-NOT FOUND}"
  if [ -z "$DATA_ROOT" ] || [ -z "$LANDMARKS" ]; then
    echo "ERROR: an input is missing (see the list above); add it with Add Input and commit again; stopping."
    exit 0
  fi
  for previous in "${PREVIOUS_RUNS[@]}"; do
    mkdir -p "$WORK/runs" && cp -rn "$previous" "$WORK/runs/" && echo "carried over runs from $previous"
  done

  if [ ! -f "$WORK/processed_v2/features.npz" ]; then
    python scripts/preprocess.py --config experiments/preprocess_v2.yaml --data-root "$DATA_ROOT" --output-dir "$WORK/processed_v2"
  fi
  # The asl-signs split must hold the participants of the shipped v2 model, so v2 and v3 are
  # compared on the same test sequences.
  python -c "import json; a = json.load(open('$WORK/processed_v2/splits_signer_independent.json'))['participants']; b = json.load(open('results/runs/cnn_signer_independent/seed_0/splits_signer_independent.json'))['participants']; assert a == b, 'split differs from v1/v2'; print('asl-signs split: same participants as v1/v2')" \
    || { echo "ERROR: the asl-signs split differs; stopping"; exit 0; }
  if [ ! -f "$WORK/processed_v3/features.npz" ]; then
    python scripts/v3_kaggle.py build-dataset --landmarks "$LANDMARKS" --kaggle-processed "$WORK/processed_v2" --out "$WORK/processed_v3"
  fi

  for SEED in 0 1 2; do
    if [ -f "$WORK/runs/e_hybrid_v3citizen/seed_$SEED/summary.json" ]; then
      echo "seed $SEED: done"
      continue
    fi
    ELAPSED=$(( ($(date +%s) - START) / 3600 ))
    if [ "$ELAPSED" -ge "$MAX_HOURS" ]; then
      echo "seed $SEED: not started after $ELAPSED h; save this version and rerun with its output attached"
      continue
    fi
    python scripts/train.py --config experiments/train_e_hybrid_v3citizen.yaml --seed "$SEED" \
      --processed-dir "$WORK/processed_v3" --output-root "$WORK/runs"
  done

  if CHOSEN=$(python scripts/v3_kaggle.py select --runs "$WORK/runs/e_hybrid_v3citizen" --out "$WORK/v3_export/selection.json"); then
    echo "chosen on validation: $CHOSEN"
    python scripts/export_web.py --run "$CHOSEN" --processed-dir "$WORK/processed_v3" --variant fp16 \
      --data-source asl-signs+asl-citizen --out "$WORK/v3_export/model" \
      --golden-dir "$WORK/v3_export/golden" --work-dir "$WORK/v3_export/work"
  fi
  echo "=== $(date -u '+%F %T') UTC: packing"
} 2>&1 | tee -a "$LOG"
cd "$REPO" 2>/dev/null && python scripts/v3_kaggle.py pack --work "$WORK" --output "$WORK/v3_bundle.zip" 2>&1 | tee -a "$LOG"
echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0, so read $LOG" | tee -a "$LOG"
exit 0
